# Tutorial 11D: Solutions - Inefficiency Determinants and Heteroscedastic Models

This notebook provides complete solutions for all exercises in Tutorial 04.

In [ ]:
# Setup
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

from scipy import stats

from panelbox.frontier import StochasticFrontier
from panelbox.frontier.utils.marginal_effects import (
    marginal_effects_bc95,
    marginal_effects_wang_2002,
)

np.random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "04_determinants"
TABLES_DIR_LATEX = BASE_DIR / "outputs" / "tables" / "latex"
TABLES_DIR_HTML = BASE_DIR / "outputs" / "tables" / "html"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR_LATEX.mkdir(parents=True, exist_ok=True)
TABLES_DIR_HTML.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

---

## Exercise 1 Solution: School Efficiency Determinants (Easy)

**Task**: Estimate a BC95 model for school data with determinants of inefficiency.

In [ ]:
# Step 1: Load and explore the school data
school_data = pd.read_csv(DATA_DIR / "school_panel.csv")

print(f"Dataset shape: {school_data.shape}")
print(f"Variables: {school_data.columns.tolist()}")
print(f"\nNumber of schools: {school_data['school_id'].nunique()}")
print(f"Number of years: {school_data['year'].nunique()}")
print(f"Year range: {school_data['year'].min()} - {school_data['year'].max()}")
print(f"\nSchool types: {school_data['school_type'].value_counts().to_dict()}")
print("\nFirst 5 observations:")
display(school_data.head())
print("\nSummary Statistics:")
display(school_data.describe().round(4))

In [ ]:
# Step 2: Create dummy variable for private schools
school_data["private"] = (school_data["school_type"] == "private").astype(int)
school_data["charter"] = (school_data["school_type"] == "charter").astype(int)

print(
    f"Private schools: {school_data['private'].sum()} observations "
    f"({school_data['private'].mean() * 100:.1f}%)"
)
print(
    f"Charter schools: {school_data['charter'].sum()} observations "
    f"({school_data['charter'].mean() * 100:.1f}%)"
)
print(
    f"Public schools:  {(1 - school_data['private'] - school_data['charter']).sum()} observations"
)

In [ ]:
# Step 3: Estimate BC95 model with inefficiency determinants
model_school = StochasticFrontier(
    data=school_data,
    depvar="log_output",
    exog=["log_teachers", "log_budget", "log_facilities"],
    inefficiency_vars=["teacher_experience", "class_size", "ses_index"],
    entity="school_id",
    time="year",
    frontier="production",
    dist="truncated_normal",
    model_type="bc95",
)

result_school = model_school.fit(method="mle", optimizer="L-BFGS-B")

print("=" * 70)
print("BC95 MODEL: SCHOOL EFFICIENCY DETERMINANTS")
print("=" * 70)
print(result_school.summary())

In [ ]:
# Step 4: Calculate and display marginal effects
try:
    me_school = result_school.marginal_effects(method="location")
except Exception as e:
    print(f"Using fallback: {e}")
    me_school = marginal_effects_bc95(result_school)

print("=" * 70)
print("MARGINAL EFFECTS ON EXPECTED INEFFICIENCY")
print("=" * 70)
display(me_school)

print("\nInterpretation of each determinant:")
for _, row in me_school.iterrows():
    var = row["variable"]
    me = row["marginal_effect"]
    pval = row["p_value"]
    sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else "n.s."))

    if "teacher_experience" in var:
        print(f"\n  Teacher Experience (ME = {me:.4f}, {sig}):")
        if me < 0:
            print(f"    One additional year of experience REDUCES inefficiency by {abs(me):.4f}")
            print("    Policy: Invest in teacher retention and professional development")
        else:
            print(f"    One additional year of experience INCREASES inefficiency by {me:.4f}")

    elif "class_size" in var:
        print(f"\n  Class Size (ME = {me:.4f}, {sig}):")
        if me > 0:
            print(f"    One additional student per class INCREASES inefficiency by {me:.4f}")
            print("    Policy: Reduce class sizes to improve efficiency")
        else:
            print(f"    One additional student per class DECREASES inefficiency by {abs(me):.4f}")

    elif "ses_index" in var:
        print(f"\n  SES Index (ME = {me:.4f}, {sig}):")
        if me < 0:
            print(f"    A 0.1-point increase in SES REDUCES inefficiency by {abs(me) * 0.1:.4f}")
            print("    Policy: Community investment programs in disadvantaged areas")
        else:
            print(f"    A 0.1-point increase in SES INCREASES inefficiency by {me * 0.1:.4f}")

In [ ]:
# Step 5: Identify most impactful determinant

# Filter out constant term and add confidence intervals
me_school = me_school[me_school["variable"] != "const"].reset_index(drop=True)
me_school["ci_lower"] = me_school["marginal_effect"] - 1.96 * me_school["std_error"]
me_school["ci_upper"] = me_school["marginal_effect"] + 1.96 * me_school["std_error"]

most_impactful_idx = me_school["marginal_effect"].abs().idxmax()
most_impactful = me_school.loc[most_impactful_idx]

print("=" * 60)
print("MOST IMPACTFUL DETERMINANT")
print("=" * 60)
print(f"Variable:        {most_impactful['variable']}")
print(f"Marginal Effect: {most_impactful['marginal_effect']:.4f}")
print(f"P-value:         {most_impactful['p_value']:.4f}")
if np.isfinite(most_impactful["ci_lower"]) and np.isfinite(most_impactful["ci_upper"]):
    print(f"95% CI:          [{most_impactful['ci_lower']:.4f}, {most_impactful['ci_upper']:.4f}]")

# Visualize marginal effects
fig, ax = plt.subplots(figsize=(10, 5))

variables = me_school["variable"].values
effects = me_school["marginal_effect"].values
std_errors = me_school["std_error"].values

colors = ["#e74c3c" if e > 0 else "#27ae60" for e in effects]
ax.barh(
    range(len(variables)),
    effects,
    color=colors,
    alpha=0.7,
    edgecolor="black",
    linewidth=0.5,
    height=0.5,
)

# Add error bars only if std_errors are finite
has_valid_se = np.isfinite(std_errors) & (std_errors > 0)
if has_valid_se.any():
    ci_lo = me_school["ci_lower"].values
    ci_hi = me_school["ci_upper"].values
    xerr_lower = np.where(has_valid_se, effects - ci_lo, 0)
    xerr_upper = np.where(has_valid_se, ci_hi - effects, 0)
    ax.errorbar(
        effects,
        range(len(variables)),
        xerr=[xerr_lower, xerr_upper],
        fmt="none",
        ecolor="black",
        capsize=5,
    )

ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(range(len(variables)))
ax.set_yticklabels(variables, fontsize=11)
ax.set_xlabel("Marginal Effect on E[u]", fontsize=12)
ax.set_title("School BC95: Marginal Effects on Inefficiency", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Additional analysis: efficiency by school type
eff_school = result_school.efficiency(estimator="bc", ci_level=0.95)
school_data["te_bc"] = eff_school["efficiency"].values

print("\nEfficiency by School Type:")
display(school_data.groupby("school_type")["te_bc"].describe().round(4))

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    x="school_type",
    y="te_bc",
    data=school_data,
    ax=ax,
    palette="Set2",
    width=0.5,
    order=["public", "charter", "private"],
)
ax.set_xlabel("School Type", fontsize=12)
ax.set_ylabel("Technical Efficiency (BC)", fontsize=12)
ax.set_title("Efficiency by School Type", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Exercise 2 Solution: Heteroscedastic Analysis (Medium)

**Task**: Estimate Wang (2002) model and compare with BC95.

In [ ]:
# Step 1: Estimate Wang (2002) model
model_wang_school = StochasticFrontier(
    data=school_data,
    depvar="log_output",
    exog=["log_teachers", "log_budget", "log_facilities"],
    inefficiency_vars=["teacher_experience", "ses_index"],  # Z: location
    het_vars=["class_size", "private"],  # W: scale
    entity="school_id",
    time="year",
    frontier="production",
    dist="truncated_normal",
)

result_wang_school = model_wang_school.fit(method="mle", optimizer="L-BFGS-B")

print("=" * 70)
print("WANG (2002): HETEROSCEDASTIC SCHOOL INEFFICIENCY")
print("=" * 70)
print(result_wang_school.summary())

In [ ]:
# Step 2: Location and scale marginal effects
print("=" * 60)
print("LOCATION MARGINAL EFFECTS (dE[u]/dz)")
print("=" * 60)

try:
    me_loc_school = result_wang_school.marginal_effects(method="location")
    display(me_loc_school)
except Exception as e:
    print(f"Location ME: {e}")
    me_loc_school = marginal_effects_wang_2002(result_wang_school, method="mean")
    display(me_loc_school)

print(f"\n{'=' * 60}")
print("SCALE MARGINAL EFFECTS (dE[u]/dw)")
print("=" * 60)

try:
    me_scale_school = result_wang_school.marginal_effects(method="scale")
    display(me_scale_school)
except Exception as e:
    print(f"Scale ME: {e}")
    try:
        me_scale_school = marginal_effects_wang_2002(result_wang_school, method="variance")
        display(me_scale_school)
    except Exception as e2:
        print(f"Scale ME fallback: {e2}")
        me_scale_school = None

print("\nInterpretation:")
print("  Location effects shift the MEAN level of inefficiency.")
print("  Scale effects change the DISPERSION of inefficiency.")
print("  A negative scale effect means the factor REDUCES variability in efficiency.")

In [ ]:
# Step 3: Model comparison
print("=" * 60)
print("MODEL COMPARISON: BC95 vs WANG (2002) FOR SCHOOLS")
print("=" * 60)

eff_wang_school = result_wang_school.efficiency(estimator="bc", ci_level=0.95)

comp_df = pd.DataFrame(
    {
        "Model": ["BC95", "Wang (2002)"],
        "Log-Likelihood": [result_school.loglik, result_wang_school.loglik],
        "AIC": [result_school.aic, result_wang_school.aic],
        "BIC": [result_school.bic, result_wang_school.bic],
        "N Params": [result_school.nparams, result_wang_school.nparams],
        "Mean TE": [eff_school["efficiency"].mean(), eff_wang_school["efficiency"].mean()],
    }
)
display(comp_df.round(4))

# Which model wins?
better_aic = "Wang (2002)" if result_wang_school.aic < result_school.aic else "BC95"
better_bic = "Wang (2002)" if result_wang_school.bic < result_school.bic else "BC95"

print(f"\nPreferred by AIC: {better_aic}")
print(f"Preferred by BIC: {better_bic}")

delta_aic = result_school.aic - result_wang_school.aic
print(f"\nDelta AIC (BC95 - Wang): {delta_aic:.2f}")
if abs(delta_aic) < 2:
    print("  Models are roughly equivalent")
elif delta_aic > 0:
    print("  Wang (2002) fits better -- heteroscedasticity matters")
else:
    print("  BC95 is sufficient -- no strong heteroscedasticity")

In [ ]:
# Step 4: Efficiency comparison scatter plot
fig, ax = plt.subplots(figsize=(8, 8))

eff_bc95_vals = eff_school["efficiency"].values
eff_wang_vals = eff_wang_school["efficiency"].values

ax.scatter(
    eff_bc95_vals,
    eff_wang_vals,
    alpha=0.3,
    s=20,
    color="steelblue",
    edgecolors="navy",
    linewidth=0.2,
)
ax.plot([0, 1], [0, 1], "r--", linewidth=2, label="45-degree line")

corr = np.corrcoef(eff_bc95_vals, eff_wang_vals)[0, 1]
rho, _ = stats.spearmanr(eff_bc95_vals, eff_wang_vals)

ax.text(
    0.05,
    0.95,
    f"Pearson r = {corr:.3f}\nSpearman rho = {rho:.3f}",
    transform=ax.transAxes,
    fontsize=12,
    verticalalignment="top",
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
)

ax.set_xlabel("Efficiency (BC95)", fontsize=13)
ax.set_ylabel("Efficiency (Wang 2002)", fontsize=13)
ax.set_title("School Efficiency: BC95 vs Wang (2002)", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean efficiency difference (Wang - BC95): {(eff_wang_vals - eff_bc95_vals).mean():.4f}")

---

## Exercise 3 Solution: Cost-Benefit Analysis (Hard)

**Task**: Rank policies by cost-effectiveness using marginal effects.

In [ ]:
# Policy definitions with costs
print("=" * 70)
print("COST-BENEFIT ANALYSIS OF SCHOOL EFFICIENCY POLICIES")
print("=" * 70)

policies = [
    {
        "name": "Teacher Development (+5 yrs experience)",
        "variable": "teacher_experience",
        "change": 5,
        "cost_per_school": 50000,
        "rationale": "Professional development programs, mentoring, retention bonuses",
    },
    {
        "name": "Class Size Reduction (-5 students)",
        "variable": "class_size",
        "change": -5,
        "cost_per_school": 100000,
        "rationale": "Hire additional teachers, build new classrooms",
    },
    {
        "name": "Community Programs (+0.1 SES)",
        "variable": "ses_index",
        "change": 0.1,
        "cost_per_school": 30000,
        "rationale": "After-school programs, parent engagement, nutrition",
    },
]

policy_results = []
for policy in policies:
    var = policy["variable"]
    me_row = me_school[me_school["variable"].str.contains(var)]

    if len(me_row) > 0:
        me = me_row["marginal_effect"].values[0]
        pval = me_row["p_value"].values[0]

        # Change in expected inefficiency
        delta_u = me * policy["change"]

        # Approximate change in efficiency (TE = exp(-u))
        # dTE/du approx = -TE * du
        mean_te = school_data["te_bc"].mean()
        approx_te_change = -mean_te * delta_u  # Positive = improvement

        # Cost-effectiveness
        cost_eff = abs(approx_te_change) / policy["cost_per_school"] * 1e6  # per $1M

        policy_results.append(
            {
                "Policy": policy["name"],
                "ME": me,
                "Change": policy["change"],
                "Delta u": delta_u,
                "Approx TE Change": approx_te_change,
                "Cost/School ($)": policy["cost_per_school"],
                "Cost-Effectiveness": cost_eff,
                "Significant": "Yes" if pval < 0.05 else "No",
            }
        )

        print(f"\n{policy['name']}:")
        print(f"  Rationale: {policy['rationale']}")
        print(f"  Marginal Effect (ME): {me:.4f} (p={pval:.4f})")
        print(f"  Change in u: {delta_u:+.4f}")
        print(f"  Approx TE change: {approx_te_change:+.4f}")
        print(f"  Cost per school: ${policy['cost_per_school']:,}")
        print(f"  Cost-effectiveness: {cost_eff:.4f} TE units per $1M")
    else:
        print(f"\n{policy['name']}: Variable {var} not found in marginal effects")

In [ ]:
# Ranking and visualization
if policy_results:
    policy_df = pd.DataFrame(policy_results)
    policy_df = policy_df.sort_values("Cost-Effectiveness", ascending=False)

    print("=" * 70)
    print("POLICY RANKING BY COST-EFFECTIVENESS")
    print("=" * 70)
    display(
        policy_df[
            ["Policy", "Approx TE Change", "Cost/School ($)", "Cost-Effectiveness", "Significant"]
        ].round(4)
    )

    # Best policy
    best = policy_df.iloc[0]
    print(f"\nRecommended Policy: {best['Policy']}")
    print(f"  Highest cost-effectiveness: {best['Cost-Effectiveness']:.4f} TE units per $1M")

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart: TE change
    colors = ["#27ae60" if x > 0 else "#e74c3c" for x in policy_df["Approx TE Change"]]
    axes[0].barh(
        range(len(policy_df)),
        policy_df["Approx TE Change"].values,
        color=colors,
        alpha=0.7,
        edgecolor="black",
        linewidth=0.5,
        height=0.5,
    )
    axes[0].set_yticks(range(len(policy_df)))
    axes[0].set_yticklabels(policy_df["Policy"].values, fontsize=10)
    axes[0].set_xlabel("Approximate TE Change", fontsize=12)
    axes[0].set_title("Efficiency Impact", fontsize=13, fontweight="bold")
    axes[0].axvline(0, color="black", linewidth=1)

    # Bar chart: Cost-effectiveness
    axes[1].barh(
        range(len(policy_df)),
        policy_df["Cost-Effectiveness"].values,
        color="steelblue",
        alpha=0.7,
        edgecolor="black",
        linewidth=0.5,
        height=0.5,
    )
    axes[1].set_yticks(range(len(policy_df)))
    axes[1].set_yticklabels(policy_df["Policy"].values, fontsize=10)
    axes[1].set_xlabel("TE units per $1M", fontsize=12)
    axes[1].set_title("Cost-Effectiveness", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()

    print("\nPolicy Discussion:")
    print("  1. The most cost-effective policy gives the most TE improvement per dollar.")
    print("  2. Statistical significance matters: only implement policies with p < 0.05.")
    print(
        "  3. Practical significance: even significant effects may be too small to justify costs."
    )
    print("  4. These are approximate calculations; real cost-benefit analysis would need")
    print("     more detailed cost modeling and general equilibrium effects.")

---

## Key Takeaways from All Exercises

1. **BC95 provides a straightforward framework** for identifying which factors drive inefficiency
2. **Marginal effects** are essential -- raw delta coefficients don't give the full picture
3. **Wang (2002)** adds value when there's evidence that inefficiency variance varies
4. **Policy simulations** bridge the gap between statistical analysis and practical recommendations
5. **Cost-benefit analysis** requires combining econometric results with cost data
6. **Model comparison** (AIC/BIC) helps choose between competing specifications